<a href="https://colab.research.google.com/github/thotasriharsha/ReinforcementLearning/blob/main/RL_ASS(6)_2159.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gymnasium as gym
import numpy as np

SEED = 42
N_EPISODES = 10          # episodes per environment
TRACE_STEPS = 5          # steps of episode 1 to print (State -> Action -> Reward -> Next State)
CLIFF_MAX_STEPS = None   # CliffWalking has no time limit by default; set e.g. 500 to cap episodes


class RandomAgent:
    """Agent that picks an action uniformly at random from the action space."""

    def __init__(self, action_space):
        self.action_space = action_space

    def act(self, state):
        return self.action_space.sample()


def describe_state(env_id, env, s):
    """Readable form of the discrete state (optional, for the trace only)."""
    if env_id.startswith("CliffWalking"):
        return f"{s} (row {s // 12}, col {s % 12})"
    if env_id.startswith("Taxi"):
        r, c, p, d = env.unwrapped.decode(s)
        return f"{s} (taxi=({r},{c}), passenger={p}, dest={d})"
    return str(s)


def make_env(env_id, max_steps=None):
    """Create the env; falls back to the other Taxi version if one is deprecated."""
    try:
        return gym.make(env_id, max_episode_steps=max_steps)
    except Exception:
        alt = "Taxi-v4" if env_id == "Taxi-v3" else "Taxi-v3"
        print(f"{env_id} unavailable in this Gymnasium version, using {alt}")
        return gym.make(alt, max_episode_steps=max_steps)


def run_random_agent(env_id, action_names, n_episodes=N_EPISODES, seed=SEED, max_steps=None):
    # Step 1: create the environment
    env = make_env(env_id, max_steps)
    env_id = env.spec.id
    env.action_space.seed(seed)
    agent = RandomAgent(env.action_space)

    print("=" * 78)
    print(f"Environment  : {env_id}")
    print(f"Observation  : {env.observation_space}")
    print(f"Action space : {env.action_space}  ->  {action_names}")   # Step 4
    print("=" * 78)

    episode_rewards, episode_lengths = [], []

    for ep in range(1, n_episodes + 1):
        # Steps 2-3: reset and observe initial state
        state, _ = env.reset(seed=seed + ep)
        total_reward, steps, done = 0, 0, False

        while not done:                                              # Step 8
            action = agent.act(state)                                # Step 5
            next_state, reward, terminated, truncated, _ = env.step(action)  # Steps 6-7
            done = terminated or truncated

            if ep == 1 and steps < TRACE_STEPS:
                print(f"t={steps + 1}: state={describe_state(env_id, env, state)} "
                      f"-> action={action} ({action_names[action]}) -> reward={reward} "
                      f"-> next_state={describe_state(env_id, env, next_state)}")

            total_reward += reward
            steps += 1
            state = next_state

        # Step 9: total and average reward for this episode
        episode_rewards.append(total_reward)
        episode_lengths.append(steps)
        print(f"Episode {ep:2d}: steps={steps:4d}  total reward={total_reward:8d}  "
              f"avg reward/step={total_reward / steps:.4f}")

    env.close()

    # Step 10: summary over all episodes
    print("-" * 78)
    print(f"Total reward over {n_episodes} episodes : {sum(episode_rewards)}")
    print(f"Average reward per episode        : {np.mean(episode_rewards):.2f}")
    print(f"Average reward per step           : {sum(episode_rewards) / sum(episode_lengths):.4f}")
    print(f"Average episode length            : {np.mean(episode_lengths):.1f}\n")


if __name__ == "__main__":
    run_random_agent("CliffWalking-v1", ["up", "right", "down", "left"], max_steps=CLIFF_MAX_STEPS)
    run_random_agent("Taxi-v3", ["south", "north", "east", "west", "pickup", "dropoff"])

Environment  : CliffWalking-v1
Observation  : Discrete(48)
Action space : Discrete(4)  ->  ['up', 'right', 'down', 'left']
t=1: state=36 (row 3, col 0) -> action=0 (up) -> reward=-1 -> next_state=24 (row 2, col 0)
t=2: state=24 (row 2, col 0) -> action=3 (left) -> reward=-1 -> next_state=24 (row 2, col 0)
t=3: state=24 (row 2, col 0) -> action=2 (down) -> reward=-1 -> next_state=36 (row 3, col 0)
t=4: state=36 (row 3, col 0) -> action=1 (right) -> reward=-100 -> next_state=36 (row 3, col 0)
t=5: state=36 (row 3, col 0) -> action=1 (right) -> reward=-100 -> next_state=36 (row 3, col 0)
Episode  1: steps=10807  total reward= -115054  avg reward/step=-10.6462
Episode  2: steps=3575  total reward=  -34958  avg reward/step=-9.7785
Episode  3: steps=2956  total reward=  -31369  avg reward/step=-10.6120
Episode  4: steps=2938  total reward=  -25708  avg reward/step=-8.7502
Episode  5: steps=14430  total reward= -138774  avg reward/step=-9.6170
Episode  6: steps=8489  total reward=  -78680  av

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(
